#  Is the Golden Leaf Still Profitable?
Seasonal Trend Analysis and Profitability in the face of climate and economic pressure (2015-2025)

**This project follows  through the sales analysis of tobacco in zimbabwe as compared against factors like climate and inflation rates
 
Key objectives:
- Analyze historical tobacco prices and sales volumes (local vs. export).
- Identify and interpret seasonal and annual trends.
- Quantify the impact of weather (rainfall, droughts) and inflation on sales and profits.
- Forecast future prices and profitability using time-series and regression models.
- Provide actionable recommendations for farmers and industry stakeholders.
- Build a clear, interactive dashboard for communicating insights.

In [1]:
#import data set

import pandas as pd


Loading Raw Data

In [2]:
df = pd.read_csv('C:/Users/user/Desktop/tobacco sales analysis datasets/tobacco_sales.csv',delimiter=';')
df.head()

,Date,Year,Day_Number,Auction_Weight_Kg,Auction_Price_USD,Contract_Weight_Kg,Contract_Price_USD,Combined_Weight_Kg,Combined_Price_USD,Prev_Year_Weight,Prev_Year_Price,Rejection_Rate_Auction,Rejection_Rate_Contract
0,22-07-2025,2025,96,19507119,3.57,328566854.0,3.31,348073973.0,3.33,228632202.0,3.43,NaN,NaN
1,21-07-2025,2025,95,19467767,3.57,328263865.0,3.31,347731632.0,3.33,228452621.0,3.43,NaN,NaN
2,18-07-2025,2025,94,19427361,3.57,327847311.0,3.32,347274672.0,3.33,228184172.0,3.43,NaN,NaN
3,17-07-2025,2025,93,19397720,3.58,326699086.0,3.32,346096806.0,3.33,227798720.0,3.43,NaN,NaN
4,16-07-2025,2025,92,19328162,3.58,325608711.0,3.32,344936873.0,3.33,227026939.0,3.43,NaN,NaN


Standardizing columns, this is because:
1. data in sormally inconsistent
2. allow for machine readability
3. comparability
4. easy analysis further down the line

In [3]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.columns.tolist())


['date', 'year', 'day_number', 'auction_weight_kg', 'auction_price_usd', 'contract_weight_kg', 'contract_price_usd', 'combined_weight_kg', 'combined_price_usd', 'prev_year_weight', 'prev_year_price', 'rejection_rate_auction', 'rejection_rate_contract']


In [4]:
df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y', errors='coerce')

In [5]:
invalid_dates = df[df['date'].isna()]
print(invalid_dates['date'])  # Should show empty (NaT) but inspect source column


Series([], Name: date, dtype: datetime64[ns])


Handling missing or inconsistent values

In [6]:
# Show number of missing values per column 
missing_counts = df.isnull().sum()
print("Missing values per column:\n", missing_counts)

Missing values per column:
 date                         0
year                         0
day_number                   0
auction_weight_kg            0
auction_price_usd           25
contract_weight_kg           2
contract_price_usd          16
combined_weight_kg         143
combined_price_usd         143
prev_year_weight           221
prev_year_price            221
rejection_rate_auction     215
rejection_rate_contract    280
dtype: int64


1. how can i use day number for analysis?
2. auction price to use mean price as a replacement but lets use mean according to year so yearly mean per missing value
3. contract weight use yearly mean as well
4. contract price we do the same as well
5. combined weight we add auction and contract weights and fill in the missing values butwe need to first confirm and check if thee are values in which the addition is not adding up and then we correct those
6. we do step 5 again on combined price
7. previous year weight and price we can drop the columns 
8. rejection rates for all we can use the online available data and use the yearly averages and fill it according to year


In [7]:
#ensuring data types are correct 
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 375 entries, 0 to 374
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   date                     375 non-null    datetime64[ns]
 1   year                     375 non-null    int64         
 2   day_number               375 non-null    int64         
 3   auction_weight_kg        375 non-null    int64         
 4   auction_price_usd        350 non-null    float64       
 5   contract_weight_kg       373 non-null    float64       
 6   contract_price_usd       359 non-null    float64       
 7   combined_weight_kg       232 non-null    float64       
 8   combined_price_usd       232 non-null    float64       
 9   prev_year_weight         154 non-null    float64       
 10  prev_year_price          154 non-null    float64       
 11  rejection_rate_auction   160 non-null    float64       
 12  rejection_rate_contract  95 non-null

**Impute Auction/Contract Prices and weights with Yearly Means
Yearly average reflects seasonal economics better than global average. You preserve inter-year variation.

In [8]:
df['auction_price_usd'] = df.groupby('year')['auction_price_usd'].transform(lambda x: x.fillna(x.mean()))
df['contract_price_usd'] = df.groupby('year')['contract_price_usd'].transform(lambda x: x.fillna(x.mean()))


In [9]:
df['contract_weight_kg'] = df.groupby('year')['contract_weight_kg'].transform(lambda x: x.fillna(x.mean()))


In [10]:
# Check for mismatches before filling
df['combined_weight_calc'] = df['auction_weight_kg'] + df['contract_weight_kg']


In [11]:
# Identify rows where combined weight is missing or mismatched
#we are taking anything that has a difference of 100kg because its the margin of error and negligable 
mask_missing_or_wrong = df['combined_weight_kg'].isna() | (abs(df['combined_weight_kg'] - df['combined_weight_calc']) > 100)

In [12]:
# Fill in
df.loc[mask_missing_or_wrong, 'combined_weight_kg'] = df.loc[mask_missing_or_wrong, 'combined_weight_calc']

In [13]:
# Drop helper column
#we are dropping because we just created it for analysis purposes it was never part of the dataset
df.drop(columns=['combined_weight_calc'], inplace=True)

**Step 4: Impute Combined Price
We'll weight it by volume:

In [14]:
def weighted_avg(row):
    if pd.notnull(row['auction_price_usd']) and pd.notnull(row['contract_price_usd']):
        return (row['auction_price_usd'] * row['auction_weight_kg'] +
                row['contract_price_usd'] * row['contract_weight_kg']) / (row['auction_weight_kg'] + row['contract_weight_kg'])
    return np.nan

df['combined_price_calc'] = df.apply(weighted_avg, axis=1)

# Replace where needed
df['combined_price_usd'] = df['combined_price_usd'].fillna(df['combined_price_calc'])

# Drop helper
df.drop(columns='combined_price_calc', inplace=True)


In [15]:
df.drop(columns=['prev_year_weight', 'prev_year_price'], inplace=True)


In [16]:
# Step 1: Define known or derived rejection rates by year
# Auction rejection rates (based on facts you provided)
rejection_rate_auction_by_year = {
    2022: 12.62,
    2023: 12.62 * 1.6078,  # 60.78% higher than 2022
    2024: 0.64,            # From your earlier input
    2025: 15.77            # As of March 27, 2025
}

# Contract rejection rates (manually informed averages)
rejection_rate_contract_by_year = {
    2022: 2.14,  # Observed low rejection rates in 2022
    2023: 1.96,  # Based on your earlier input
    2024: 1.20,  # Approximate average
    2025: 2.14   # Known from early 2025 report
}

# Step 2: Fill missing values in auction rejection rates
for year, rate in rejection_rate_auction_by_year.items():
    mask = (df['year'] == year) & (df['rejection_rate_auction'].isna())
    df.loc[mask, 'rejection_rate_auction'] = rate

# Step 3: Fill missing values in contract rejection rates
for year, rate in rejection_rate_contract_by_year.items():
    mask = (df['year'] == year) & (df['rejection_rate_contract'].isna())
    df.loc[mask, 'rejection_rate_contract'] = rate

# Step 4 (Optional): Check how many are still missing
print("✅ Remaining missing values after filling:")
print(df[['rejection_rate_auction', 'rejection_rate_contract']].isna().sum())


✅ Remaining missing values after filling:
rejection_rate_auction     0
rejection_rate_contract    0
dtype: int64


In [17]:
df.describe()

,year,day_number,auction_weight_kg,auction_price_usd,contract_weight_kg,contract_price_usd,combined_weight_kg,combined_price_usd,rejection_rate_auction,rejection_rate_contract
count,375.00000,375.000000,3.750000e+02,375.000000,3.750000e+02,375.000000,3.750000e+02,375.000000,375.000000,375.000000
mean,2023.58400,57.434667,1.077175e+07,3.260194,1.613189e+08,3.232396,1.720907e+08,3.233728,9.580189,1.813280
std,1.05352,31.944979,5.927428e+06,0.325334,9.022190e+07,0.221402,9.587379e+07,0.223745,5.964517,0.407962
min,2022.00000,1.000000,4.893800e+04,2.510000,2.094650e+05,2.620000,3.932790e+05,2.579200,0.640000,1.200000
25%,2023.00000,31.000000,5.909175e+06,2.900000,8.701016e+07,3.020000,9.288555e+07,3.020155,0.640000,1.200000
50%,2024.00000,56.000000,1.171873e+07,3.220000,1.772753e+08,3.320000,1.886521e+08,3.330000,11.550000,1.960000
75%,2024.00000,84.000000,1.482156e+07,3.590000,2.181359e+08,3.430000,2.316545e+08,3.435000,14.490000,2.140000
max,2025.00000,122.000000,1.991368e+07,3.770000,3.285669e+08,3.590000,3.480740e+08,3.580000,20.290436,2.690000


In [18]:
cols_to_check = [
    'auction_price_usd', 'contract_price_usd', 'combined_price_usd',
    'auction_weight_kg', 'contract_weight_kg', 'combined_weight_kg',
    'rejection_rate_auction', 'rejection_rate_contract'
]


In [19]:
df[cols_to_check].describe().T


,count,mean,std,min,25%,50%,75%,max
auction_price_usd,375.0,3.260194e+00,3.253339e-01,2.5100,2.900000e+00,3.220000e+00,3.590000e+00,3.770000e+00
contract_price_usd,375.0,3.232396e+00,2.214019e-01,2.6200,3.020000e+00,3.320000e+00,3.430000e+00,3.590000e+00
combined_price_usd,375.0,3.233728e+00,2.237446e-01,2.5792,3.020155e+00,3.330000e+00,3.435000e+00,3.580000e+00
auction_weight_kg,375.0,1.077175e+07,5.927428e+06,48938.0000,5.909175e+06,1.171873e+07,1.482156e+07,1.991368e+07
contract_weight_kg,375.0,1.613189e+08,9.022190e+07,209465.0000,8.701016e+07,1.772753e+08,2.181359e+08,3.285669e+08
combined_weight_kg,375.0,1.720907e+08,9.587379e+07,393279.0000,9.288555e+07,1.886521e+08,2.316545e+08,3.480740e+08
rejection_rate_auction,375.0,9.580189e+00,5.964517e+00,0.6400,6.400000e-01,1.155000e+01,1.449000e+01,2.029044e+01
rejection_rate_contract,375.0,1.813280e+00,4.079624e-01,1.2000,1.200000e+00,1.960000e+00,2.140000e+00,2.690000e+00


In [20]:
outlier_flags = {}

for col in cols_to_check:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Flag the rows with outliers
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_flags[col] = len(outliers)
    print(f"{col}: {len(outliers)} outliers detected")

    # Optional: Add a new column to mark outliers
    df[f'{col}_outlier'] = ((df[col] < lower_bound) | (df[col] > upper_bound))


auction_price_usd: 0 outliers detected
contract_price_usd: 0 outliers detected
combined_price_usd: 0 outliers detected
auction_weight_kg: 0 outliers detected
contract_weight_kg: 0 outliers detected
combined_weight_kg: 0 outliers detected
rejection_rate_auction: 0 outliers detected
rejection_rate_contract: 0 outliers detected


**What Is Feature Engineering?
- Feature engineering is the process of:
- Creating new variables (features) or transforming existing ones to help your model or analysis better understand the underlying patterns.

In [21]:
#creating time based features 
df['month'] = df['date'].dt.month
df['week'] = df['date'].dt.isocalendar().week
df['quarter'] = df['date'].dt.quarter

In [22]:
# Create Price & Weight Gaps

# Price difference between auction and contract
df['price_gap_ac'] = df['auction_price_usd'] - df['contract_price_usd']

# Weight difference between auction and contract
df['weight_gap_ac'] = df['auction_weight_kg'] - df['contract_weight_kg']


In [23]:
# Create Rejection Burden to assess market stress

# Combined rejection influence (optional)
df['total_rejection_pressure'] = df['rejection_rate_auction'] * df['auction_weight_kg'] + \
                                  df['rejection_rate_contract'] * df['contract_weight_kg']


In [28]:
# 7-day rolling average of combined price
#This smooths volatility and lets you analyze market momentum or decay — key in forecasting and stakeholder analysis.

df['rolling_avg_price'] = df['combined_price_usd'].rolling(window=7, min_periods=1).mean()

# 14-day rolling rejection rate
df['rolling_rejection_rate'] = (
    df[['rejection_rate_auction', 'rejection_rate_contract']]
    .mean(axis=1)
    .rolling(window=14, min_periods=1)
    .mean()
)


In [29]:
df.to_csv('cleaned_data.csv', index=False)


In [30]:
import os

print(os.getcwd())  # shows current directory

print(os.listdir())  # shows files/folders in current directory


C:\Users\user
['-1.14-windows.xml', '.android', '.bash_history', '.cache', '.cisco', '.conda', '.condarc', '.continuum', '.cursor', '.dotnet', '.emulator_console_auth_token', '.gitconfig', '.gradle', '.idlerc', '.ipynb_checkpoints', '.ipython', '.jupyter', '.lesshst', '.matplotlib', '.nbi', '.popsql.json', '.redhat', '.ssh', '.vscode', 'AppData', 'Application Data', 'assignment steps.ipynb', 'cleaned_data.csv', 'Contacts', 'Cookies', 'Desktop', 'Documents', 'Downloads', 'Drug Addiction .ipynb', 'edb_npgsql.exe', 'edb_npgsql.exe-20250117171653', 'edb_pgagent_pg17.exe', 'edb_psqlodbc.exe', 'edb_sqlprofiler_pg17.exe', 'Favorites', 'go', 'IntelGraphicsProfiles', 'Iris ML Tutorial.ipynb', 'job allocation.jpg', 'Links', 'Local Settings', 'Microsoft', 'Music', 'My Documents', 'NCH Software Suite', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{cd05bfae-b5b1-11ef-bae4-f37208687570}.TM.blf', 'NTUSER.DAT{cd05bfae-b5b1-11ef-bae4-f37208687570}.TMContainer000000000000000

In [31]:
cpi_df = pd.read_excel("C:\\Users\\user\\Desktop\\tobacco sales analysis datasets\\Consumer Price Index July 2020.xlsx")
cpi_df.head()

,INDICATOR,Units,Jul 2023,Aug 2023,Sep 2023,Oct 2023,Nov 2023,Dec 2023,Jan 2024,Feb 2024,...,Oct 2024,Nov 2024,Dec 2024,Jan 2025,Feb 2025,Mar 2025,Apr 2025,May 2025,Jun 2025,Jul 2025
0,All items,Index,73.818565,72.828456,73.523897,75.330699,78.751940,82.455074,87.881952,92.618198,...,108.737454,111.153373,112.391490,125.397847,125.783244,125.844450,126.183563,126.210220,126.073103,126.819799
1,Food and non-alcoholic beverages,Index,60.698711,59.587050,60.214851,61.670283,64.682909,70.268774,80.816228,88.761574,...,119.100329,125.320400,128.881715,145.920966,146.561882,146.524440,146.369473,146.344225,145.541548,145.336647
2,Alcoholic beverages and tobacco,Index,85.125488,84.179538,84.452673,86.063408,87.732506,90.795527,93.197688,96.280030,...,108.145330,110.149829,110.294688,125.108179,125.868445,126.263189,127.280202,127.255060,127.427220,127.761408
3,Clothing and footwear,Index,97.894754,97.789834,97.831307,98.910200,99.503638,100.905038,98.347715,98.601793,...,101.295800,101.726562,102.102508,113.679543,114.326550,114.757877,115.209240,115.038054,114.949386,114.760444
4,Housing water electricity gas and other fuels,Index,70.172223,68.766056,68.997751,72.420553,78.886959,83.147346,87.706722,91.951268,...,104.556611,105.115014,105.471621,121.195802,121.257177,121.276315,122.232786,122.288653,122.362429,125.076411


In [32]:
# Melt the monthly columns to rows
cpi_long = cpi_df.melt(
    id_vars=["INDICATOR", "Units"],
    var_name="Month_Year",
    value_name="CPI"
)


In [33]:
# Add a day (1st) to each Month_Year so i can parse it as a date
cpi_long["Month_Year"] = "1 " + cpi_long["Month_Year"]

# Now convert to datetime
cpi_long["Date"] = pd.to_datetime(cpi_long["Month_Year"], format="%d %b %Y")


In [34]:
cpi_long["Year"] = cpi_long["Date"].dt.year
cpi_long["Month"] = cpi_long["Date"].dt.month


In [35]:
cpi_clean = cpi_long.drop(columns=["Units", "Month_Year"])
cpi_clean.head()

,INDICATOR,CPI,Date,Year,Month
0,All items,73.818565,2023-07-01,2023,7
1,Food and non-alcoholic beverages,60.698711,2023-07-01,2023,7
2,Alcoholic beverages and tobacco,85.125488,2023-07-01,2023,7
3,Clothing and footwear,97.894754,2023-07-01,2023,7
4,Housing water electricity gas and other fuels,70.172223,2023-07-01,2023,7


In [36]:
gdp_df=pd.read_excel("C:\\Users\\user\\Desktop\\tobacco sales analysis datasets\\GDP at Constant Prices Billion ZWL at 2019 Prices & Annual Growth Rates.xlsx", sheet_name='Data')
gdp_df.head()

,INDICATOR,2017,2018,2019,2020,2021,2022,2023,2024
0,"GDP at Constant Prices, Billion ZWL at 2019 Pr...",215.71169,226.518558,150984.728785,139182.327095,150968.310087,160236.652090,168788.050331,68.735890
1,"GDP at Constant Prices, Annual Growth Rates (r...",3.44700,5.009867,-6.330000,-7.816951,8.468017,6.139263,5.336730,-99.959277


In [37]:
exchange_rate_df= pd.read_csv("C:\\Users\\user\\Desktop\\tobacco sales analysis datasets\\month-month-exchange-rate-2020-2024.csv", delimiter=';')
exchange_rate_df.head()

,Domain Code,Domain,Area Code (M49),Area,ISO Currency Code (FAO),Currency,Element Code,Element,Year Code,Year,Months Code,Months,Unit,Value,Flag,Flag Description
0,PE,Exchange rates,716,Zimbabwe,ZWL,Zimbabwe Dollar,LCU,Local currency units per USD,2020,2020,7021,Annual value,NaN,51.329013,X,Figure from international organizations
1,PE,Exchange rates,716,Zimbabwe,ZWL,Zimbabwe Dollar,LCU,Local currency units per USD,2020,2020,7001,January,NaN,17.095014,X,Figure from international organizations
2,PE,Exchange rates,716,Zimbabwe,ZWL,Zimbabwe Dollar,LCU,Local currency units per USD,2020,2020,7002,February,NaN,17.682317,X,Figure from international organizations
3,PE,Exchange rates,716,Zimbabwe,ZWL,Zimbabwe Dollar,LCU,Local currency units per USD,2020,2020,7003,March,NaN,21.160000,X,Figure from international organizations
4,PE,Exchange rates,716,Zimbabwe,ZWL,Zimbabwe Dollar,LCU,Local currency units per USD,2020,2020,7004,April,NaN,25.000000,X,Figure from international organizations


In [38]:
exchange_rate_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63 entries, 0 to 62
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Domain Code              63 non-null     object 
 1   Domain                   63 non-null     object 
 2   Area Code (M49)          63 non-null     int64  
 3   Area                     63 non-null     object 
 4   ISO Currency Code (FAO)  63 non-null     object 
 5   Currency                 63 non-null     object 
 6   Element Code             63 non-null     object 
 7   Element                  63 non-null     object 
 8   Year Code                63 non-null     int64  
 9   Year                     63 non-null     int64  
 10  Months Code              63 non-null     int64  
 11  Months                   63 non-null     object 
 12  Unit                     0 non-null      float64
 13  Value                    63 non-null     float64
 14  Flag                     63 

In [45]:
# Drop unnecessary columns
df_exchange_rate_new = exchange_rate_df[['Year', 'Months', 'Value']]


In [ ]:
#